# Huấn luyện Emotion Classifier (6 nhãn) — Colab / Kaggle
Model: **ViSoBERT**. Dataset: `emotion_6label_data.csv`.


In [2]:
!pip install transformers datasets evaluate accelerate scikit-learn pyvi -q


In [ ]:
import pandas as pd
import numpy as np
import torch
import re
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset, DatasetDict
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
from pyvi import ViTokenizer

# ================= CẤU HÌNH =================
# 'uitnlp/visobert'               - SOTA social VN, không cần tách từ
# 'vinai/phobert-base-v2'          - Baseline mạnh, cần tách từ
# 'bkai-foundation-models/BamiBERT-base'  - SOTA mới, không cần tách từ
MODEL_NAME = 'uitnlp/visobert'
DATA_PATH  = 'Data/active_inputs/emotion_training.csv'
MAX_LEN    = 128


In [4]:
# 1. Đọc và Tiền xử lý dữ liệu
df = pd.read_csv(DATA_PATH)

# 7 nhãn cốt lõi
label_list = ['vui', 'buon', 'lo_au', 'gian_du', 'trung_tinh', 'hy_vong', 'that_vong']
label2id = {l: i for i, l in enumerate(label_list)}
id2label  = {i: l for i, l in enumerate(label_list)}

# ── Emoji/pattern lists: CHỈ dùng để QC phát hiện nhãn nghi ngờ (cell 8b),
# KHÔNG áp dụng lên text đưa vào model. Lý do: train và inference phải nhận
# CÙNG một phân phối text — nếu strip không nhất quán 2 phía sẽ lệch train/
# inference; nếu strip nhất quán cả 2 phía thì lại xóa mất tín hiệu cảm xúc
# thật (ViSoBERT thiết kế cho social-media text có emoji, emoji là tín hiệu
# hợp lệ trong phần lớn trường hợp, đã xác nhận bằng phân tích trước đó).
POS_PATTERNS = [
    r'(?<!\w)hihi+(?!\w)', r'(?<!\w)hehe+(?!\w)', r'(?<!\w)haha+(?!\w)',
    r'(?<!\w)k+k+k+(?!\w)', r':\)+', r':D+', r':v+', r'=\)+',
]
NEG_PATTERNS = [
    r'(?<!\w)hu+hu+(?!\w)', r'(?<!\w)hic+(?!\w)',
    r':\(+', r'=\(+', r'T_T+',
]
# Chỉ giữ emoji có cực tính rõ ràng — bỏ 😏🤡😶😑 (mỉa mai/trung tính/phụ thuộc ngữ cảnh)
POS_EMOJIS = [
    '😂','🤣','😁','😀','😊','😄','😆','😅','🤪','😜','😋','🥰','😍','🤩',
    '😎','😌','🙃','😇','🤗','❤️','💕','💖','💗','💓','💞','💘','💝',
    '🎉','🎊','👏','✨','🌟','⭐','🥳','😙','😚','😗','😘',
]
NEG_EMOJIS = [
    '😭','😢','😞','😔','😟','😕','🙁','☹️','😫','😩','🥺',
    '😡','🤬','😠','😤','💔','😰','😨','😱','😖','😣','😓','😥',
    '🤧','😪','🥵','🥀',
]
NEGATIVE_LABELS = {'buon', 'lo_au', 'gian_du', 'that_vong'}
POSITIVE_LABELS = {'vui', 'hy_vong'}

def strip_contradictory_signal(text, label):
    """CHỈ dùng cho QC (classify_conflict ở cell 8b) — xóa icon/pattern NGƯỢC
    với nhãn để xem phần chữ còn lại có còn khớp nhãn không. KHÔNG dùng để
    tiền xử lý text đưa vào model (train lẫn inference dùng nguyên văn)."""
    text = str(text)
    if label in NEGATIVE_LABELS:
        for emoji in POS_EMOJIS:
            text = text.replace(emoji, '')
        for p in POS_PATTERNS:
            text = re.sub(p, '', text, flags=re.IGNORECASE)
    if label in POSITIVE_LABELS:
        for emoji in NEG_EMOJIS:
            text = text.replace(emoji, '')
        for p in NEG_PATTERNS:
            text = re.sub(p, '', text, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', text).strip()

df['label'] = df['emotion'].map(label2id)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

# ✅ FIX (bug train/inference lệch phân phối): KHÔNG strip emoji/pattern khỏi
# text đưa vào model nữa. Train và inference giờ dùng NGUYÊN VĂN (chỉ chuẩn
# hóa khoảng trắng) — nhất quán 2 phía, giữ tín hiệu emoji cho model.
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'] != '']

# Tách từ chỉ khi dùng PhoBERT
if 'phobert' in MODEL_NAME.lower():
    print('Tách từ (Word Segmentation) cho PhoBERT...')
    df['text'] = df['text'].apply(lambda x: ViTokenizer.tokenize(str(x)))
else:
    df['text'] = df['text'].astype(str)

df = df[['text', 'label']]
print(df['label'].value_counts().rename(id2label))


label
gian_du       1500
vui           1500
lo_au         1500
buon          1500
that_vong     1500
hy_vong       1500
trung_tinh    1250
Name: count, dtype: int64


In [5]:
# 2. Stratified Split 80-10-10
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

hg_dataset = DatasetDict({
    'train':      Dataset.from_pandas(train_df, preserve_index=False),
    'validation': Dataset.from_pandas(val_df,   preserve_index=False),
    'test':       Dataset.from_pandas(test_df,  preserve_index=False),
})


Train: 8200 | Val: 1025 | Test: 1025


In [6]:
# 3. Tokenization
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, max_length=MAX_LEN)

tokenized = hg_dataset.map(tokenize_fn, batched=True)
collator  = DataCollatorWithPadding(tokenizer=tokenizer)


Map:   0%|          | 0/8200 [00:00<?, ? examples/s]

Map:   0%|          | 0/1025 [00:00<?, ? examples/s]

Map:   0%|          | 0/1025 [00:00<?, ? examples/s]

In [7]:
# 4. Metrics
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    base_metrics = {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='macro'),
        'precision': precision_score(labels, preds, average='macro', zero_division=0),
        'recall': recall_score(labels, preds, average='macro', zero_division=0)
    }
    # Tính Top-2 Accuracy
    top2_preds = np.argsort(logits, axis=1)[:, -2:]
    top2_hits = [1 if labels[i] in top2_preds[i] else 0 for i in range(len(labels))]
    base_metrics['top2_accuracy'] = sum(top2_hits) / len(labels)
    return base_metrics


In [8]:
# 5. Model + Class Weights tự động
import torch.nn as nn

cw = compute_class_weight('balanced', classes=np.unique(df['label']), y=df['label'])
cw_tensor = torch.tensor(cw, dtype=torch.float32).to('cuda' if torch.cuda.is_available() else 'cpu')
print('Class weights:', dict(zip(label_list, cw.round(3))))

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get('labels')
        outputs = model(**inputs)
        logits  = outputs.get('logits')
        loss    = nn.CrossEntropyLoss(weight=cw_tensor, label_smoothing=0.1)(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=7, id2label=id2label, label2id=label2id
)


Class weights: {'vui': np.float64(0.976), 'buon': np.float64(0.976), 'lo_au': np.float64(0.976), 'gian_du': np.float64(0.976), 'trung_tinh': np.float64(1.171), 'hy_vong': np.float64(0.976), 'that_vong': np.float64(0.976)}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: uitnlp/visobert
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
# 6. Training Arguments
import inspect

training_args = TrainingArguments(
    output_dir               = './emotion_model_results',
    learning_rate            = 1e-5,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    num_train_epochs         = 7,
    weight_decay             = 0.05,
    warmup_ratio             = 0.15,
    lr_scheduler_type        = 'cosine',
    eval_strategy            = 'epoch',
    save_strategy            = 'epoch',
    load_best_model_at_end   = True,
    metric_for_best_model    = 'f1',
    greater_is_better        = True,
    fp16                     = True,
    report_to                = 'none',
)

sig = inspect.signature(Trainer.__init__)
kwargs = dict(
    model          = model,
    args           = training_args,
    train_dataset  = tokenized['train'],
    eval_dataset   = tokenized['validation'],
    data_collator  = collator,
    compute_metrics= compute_metrics,
)
kwargs['processing_class' if 'processing_class' in sig.parameters else 'tokenizer'] = tokenizer
trainer = CustomTrainer(**kwargs)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [10]:
# 7. Train 🚀
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Top2 Accuracy
1,No log,1.555997,0.466341,0.444914,0.472213,0.466667,0.657561
2,1.639979,1.325869,0.580488,0.583048,0.602763,0.584381,0.746341
3,1.639979,1.309559,0.604878,0.604210,0.613539,0.606857,0.782439
4,1.104416,1.346954,0.619512,0.614741,0.616329,0.619238,0.787317
5,1.104416,1.382806,0.611707,0.608765,0.609526,0.610667,0.793171
6,0.872039,1.395336,0.615610,0.613733,0.615494,0.614857,0.787317
7,0.872039,1.402421,0.614634,0.611449,0.612151,0.614095,0.786341


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1799, training_loss=1.1379873414646593, metrics={'train_runtime': 387.0143, 'train_samples_per_second': 148.315, 'train_steps_per_second': 4.648, 'total_flos': 3095920100654400.0, 'train_loss': 1.1379873414646593, 'epoch': 7.0})

In [11]:
# 8. Evaluate on Test
print('=== KẾT QUẢ TẬP TEST ===')
results = trainer.evaluate(tokenized['test'])
for k, v in results.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')


=== KẾT QUẢ TẬP TEST ===


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall,Top2 Accuracy
0.872039,1.370322,7,0.593171,0.586787,0.588227,0.591619,0.772683


eval_loss: 1.3703
eval_accuracy: 0.5932
eval_f1: 0.5868
eval_precision: 0.5882
eval_recall: 0.5916
eval_top2_accuracy: 0.7727


In [12]:
# 8a. K-Fold Out-of-Fold predictions (Bước 2-4 của method.md)
# Train 5 model tạm, mỗi model chấm fold nó CHƯA TỪNG học -> pred_probs khách quan
# cho toàn bộ dataset. ~5 x 6-7 phút GPU. Kết quả lưu oof_pred_probs.npy để 8b dùng.
from sklearn.model_selection import StratifiedKFold

K_FOLDS = 5

rv = pd.read_csv(DATA_PATH).reset_index().rename(columns={'index': 'id'})
rv['label'] = rv['emotion'].map(label2id)
rv = rv.dropna(subset=['label']).reset_index(drop=True)
rv['label'] = rv['label'].astype(int)
rv['emotion_current'] = rv['label'].map(id2label)
# raw_text = TEXT THỰC TẾ ĐƯA VÀO MODEL — phải khớp CHÍNH XÁC cách xử lý ở cell 3
# (nguyên văn, chỉ chuẩn hóa khoảng trắng), để OOF probs phản ánh đúng model thật
# sẽ được train (không strip emoji theo nhãn — xem comment ở cell 3).
rv['raw_text'] = rv['text'].astype(str).str.strip()
rv = rv[rv['raw_text'] != ''].reset_index(drop=True)
rv['clean_text_'] = rv['raw_text']  # giữ tên cột để 8b dùng lại, nội dung = raw_text (không strip)

oof_probs = np.zeros((len(rv), len(label_list)), dtype=np.float32)
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

fold_args = TrainingArguments(
    output_dir='./kfold_tmp', learning_rate=1e-5,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    num_train_epochs=3,          # 3 epoch đủ cho máy dò (không cần model tốt nhất)
    weight_decay=0.05, warmup_ratio=0.15, lr_scheduler_type='cosine',
    eval_strategy='no', save_strategy='no',   # không lưu checkpoint -> nhẹ đĩa
    fp16=True, report_to='none',
)

for fold, (tr_idx, va_idx) in enumerate(skf.split(rv['clean_text_'], rv['label'])):
    print(f'===== Fold {fold+1}/{K_FOLDS}: train {len(tr_idx)} | chấm {len(va_idx)} =====')
    ds_tr = Dataset.from_pandas(rv.iloc[tr_idx][['clean_text_', 'label']].rename(columns={'clean_text_': 'text'}), preserve_index=False).map(tokenize_fn, batched=True)
    ds_va = Dataset.from_pandas(rv.iloc[va_idx][['clean_text_', 'label']].rename(columns={'clean_text_': 'text'}), preserve_index=False).map(tokenize_fn, batched=True)

    fold_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=7, id2label=id2label, label2id=label2id)
    fold_trainer = CustomTrainer(model=fold_model, args=fold_args,
                                 train_dataset=ds_tr, data_collator=collator,
                                 **{('processing_class' if 'processing_class' in sig.parameters else 'tokenizer'): tokenizer})
    fold_trainer.train()
    logits = fold_trainer.predict(ds_va).predictions
    oof_probs[va_idx] = torch.nn.functional.softmax(torch.tensor(logits), dim=1).numpy()

    del fold_model, fold_trainer
    torch.cuda.empty_cache()

np.save('Data/pipeline_artifacts/oof_pred_probs.npy', oof_probs)
rv.to_csv('Data/pipeline_artifacts/rv_meta.csv', index=False)
print(f'\nXong: oof_pred_probs.npy shape={oof_probs.shape} (out-of-fold 100%)')


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


===== Fold 1/5: train 8200 | chấm 2050 =====


Map:   0%|          | 0/8200 [00:00<?, ? examples/s]

Map:   0%|          | 0/2050 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Map:   0%|          | 0/2050 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: uitnlp/visobert
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,1.553801



Xong: oof_pred_probs.npy shape=(10250, 7) (out-of-fold 100%)


In [13]:
# 8b. Cleanlab (OOF) + Heuristic icon/chữ -> review_candidates.csv (schema method.md)
!pip install cleanlab -q

from cleanlab.filter import find_label_issues
from cleanlab.rank import get_label_quality_scores

# Nạp kết quả K-fold từ cell 8a (bắt buộc chạy 8a trước)
oof_probs = np.load('Data/pipeline_artifacts/oof_pred_probs.npy')
rv = pd.read_csv('Data/pipeline_artifacts/rv_meta.csv')
assert len(rv) == len(oof_probs), 'rv_meta.csv và oof_pred_probs.npy lệch nhau — chạy lại cell 8a'

# ── Heuristic: icon nói 1 đằng, phần chữ (sau khi bỏ icon NGƯỢC nhãn) nói 1 nẻo ──
# strip_contradictory_signal CHỈ dùng ở đây (QC) — không phải text đưa vào model.
POS_CUES = ['vui', 'hạnh phúc', 'tuyệt vời', 'yêu đời', 'sung sướng', 'mừng quá', 'tự hào']
NEG_CUES = ['buồn', 'hối hận', 'chán', 'thất vọng', 'tuyệt vọng', 'đau lòng', 'khóc',
            'cô đơn', 'bế tắc', 'tự ti', 'mệt mỏi', 'lo lắng', 'sợ hãi', 'tức giận', 'ghét']

rv['qc_text'] = rv.apply(lambda r: strip_contradictory_signal(r['raw_text'], r['emotion_current']), axis=1)

def classify_conflict(raw, qc_text, label):
    has_pos_icon = any(e in raw for e in POS_EMOJIS)
    has_neg_icon = any(e in raw for e in NEG_EMOJIS)
    if not has_pos_icon and not has_neg_icon:
        return None
    if len(str(qc_text)) < 3:
        return 'icon_only'            # icon là tín hiệu DUY NHẤT -> ứng viên remove
    low = str(qc_text).lower()
    has_pos_word = any(w in low for w in POS_CUES)
    has_neg_word = any(w in low for w in NEG_CUES)
    if label in POSITIVE_LABELS and has_neg_word and not has_pos_word:
        return 'label_theo_icon_pos'  # nhãn tích cực nhưng chữ thuần tiêu cực -> ứng viên change
    if label in NEGATIVE_LABELS and has_pos_word and not has_neg_word:
        return 'label_theo_icon_neg'
    return None

rv['icon_flag'] = rv.apply(lambda r: classify_conflict(r['raw_text'], r['qc_text'], r['emotion_current']), axis=1)
print('=== Heuristic icon/chữ ===')
print(rv['icon_flag'].value_counts(dropna=False))

# ── Cleanlab trên OOF probs: normalized_margin làm điểm ranking chính ──
labels_arr = rv['label'].values
rv['label_quality_score'] = get_label_quality_scores(labels_arr, oof_probs, method='normalized_margin')
rv['predicted_label'] = [id2label[i] for i in oof_probs.argmax(axis=1)]
rv['original_label_probability'] = oof_probs[np.arange(len(rv)), labels_arr]  # thông tin giải thích phụ

issue_idx = find_label_issues(labels_arr, oof_probs, return_indices_ranked_by='normalized_margin')
rv['cleanlab_suspect'] = False
rv.loc[issue_idx, 'cleanlab_suspect'] = True
print(f'\n=== Cleanlab (OOF): {len(issue_idx)}/{len(rv)} mẫu nghi ngờ sai nhãn ===')

# ── Xuất TOÀN BỘ danh sách nghi ngờ (không cap) -> không mất dấu vết ──
# Dùng để: (1) audit đầy đủ, (2) duyệt tiếp đợt 2/3 sau này mà không cần
# chạy lại K-fold, vì mọi thứ đã tính xong và lưu lại ở đây.
all_suspects = rv[rv['icon_flag'].notna() | rv['cleanlab_suspect']].copy()
all_suspects = all_suspects.sort_values('label_quality_score')
all_suspects = all_suspects.rename(columns={'raw_text': 'text_raw', 'emotion_current': 'original_label'})
all_cols = ['id', 'text_raw', 'original_label', 'predicted_label',
            'original_label_probability', 'label_quality_score',
            'icon_flag', 'cleanlab_suspect']
all_suspects[all_cols].to_csv('Data/pipeline_artifacts/all_suspects_full.csv', index=False)
print(f'Đã lưu TOÀN BỘ {len(all_suspects)} dòng nghi ngờ -> all_suspects_full.csv (không mất, không cap)')

# ── Gộp 2 nguồn -> shortlist duyệt tay đợt 1 (top 300-500 theo method.md) ──
MAX_REVIEW = 500
suspects = all_suspects  # đã sort theo score
icon_rows  = suspects[suspects['icon_flag'].notna()]
other_rows = suspects[suspects['icon_flag'].isna()].head(max(0, MAX_REVIEW - len(icon_rows)))
review = pd.concat([icon_rows, other_rows]).sort_values('label_quality_score')

# Schema đúng method.md + pre-fill action cho nhóm khách quan
review = review.copy()
review['action'] = review['icon_flag'].apply(lambda f: 'remove' if f == 'icon_only' else '')
review['new_label'] = ''
review['review_note'] = ''

out_cols = ['id', 'text_raw', 'original_label', 'predicted_label',
            'original_label_probability', 'label_quality_score',
            'icon_flag', 'action', 'new_label', 'review_note']
review[out_cols].to_csv('Data/pipeline_artifacts/review_candidates.csv', index=False)
print(f'Xuất {len(review)}/{len(all_suspects)} dòng (đợt 1, ưu tiên nghi ngờ nhất) -> review_candidates.csv')
print('Duyệt tay: action = keep (hoặc để trống) / change (điền new_label) / remove')
print(f'\nCòn lại {len(all_suspects) - len(review)} dòng chưa duyệt trong all_suspects_full.csv'
      f' — muốn duyệt đợt 2 thì lọc tiếp từ file đó theo id, không cần chạy lại 8a/8b.')
review[out_cols].head(20)


=== Heuristic icon/chữ ===
icon_flag
NaN                    10201
label_theo_icon_pos       37
icon_only                  6
label_theo_icon_neg        6
Name: count, dtype: int64

=== Cleanlab (OOF): 3206/10250 mẫu nghi ngờ sai nhãn ===
Đã lưu TOÀN BỘ 3248 dòng nghi ngờ -> all_suspects_full.csv (không mất, không cap)
Xuất 500/3248 dòng (đợt 1, ưu tiên nghi ngờ nhất) -> review_candidates.csv
Duyệt tay: action = keep (hoặc để trống) / change (điền new_label) / remove

Còn lại 2748 dòng chưa duyệt trong all_suspects_full.csv — muốn duyệt đợt 2 thì lọc tiếp từ file đó theo id, không cần chạy lại 8a/8b.


,id,text_raw,original_label,predicted_label,original_label_probability,label_quality_score,icon_flag,action,new_label,review_note
4687,4687,"ngoại hình t mập lắm , làm j cũng rón rén , ng...",that_vong,lo_au,0.014301,0.042173,NaN,,,
8554,8554,"per đang định đi chụp , thấy cái này sợ quá đé...",that_vong,lo_au,0.015079,0.043253,NaN,,,
9051,9051,mũi to cx sợ lắm mặt t ko có nét lúc cx ngại m...,buon,lo_au,0.014350,0.044139,NaN,,,
2527,2527,Thế giới này thật đáng sợ 🙄,that_vong,lo_au,0.015107,0.046698,NaN,,,
7851,7851,"Châu Minh Melina Hosten sợ luôn, lại còn gửi b...",that_vong,lo_au,0.014274,0.047450,NaN,,,
7496,7496,khi nào vụ này kết thúc thì mình uống cf trung...,that_vong,lo_au,0.019391,0.047994,NaN,,,
582,582,có ai 9 năm không dám chụp với ai 1 tấm vì sợ ...,buon,lo_au,0.008389,0.049211,NaN,,,
2524,2524,Mong pháp luật xử đúng tội của bọn ác nhân,hy_vong,gian_du,0.005904,0.049573,NaN,,,
5593,5593,Sợ cái nết vậy rồi nên khó thay đổi đc á bạn.,that_vong,lo_au,0.024910,0.050471,NaN,,,
8914,8914,hồi lớp 1 đú theo lũ bạn về học nấu . thế là c...,vui,lo_au,0.011375,0.051235,NaN,,,


In [ ]:
# 8b-bis. Gắn cờ MÂU THUẪN vs CHỒNG LẤN HỢP LỆ -> ưu tiên duyệt đúng chỗ
# Không đụng vào action/new_label bạn đã điền (chỉ thêm cột + sắp lại thứ tự).
#
# Ý tưởng: original_label + predicted_label CHỒNG LẤN HỢP LỆ (vd vui/hy_vong,
# buon/that_vong) thường chỉ là 2 cảm xúc tự nhiên đi cùng nhau -> ít cần sửa.
# original_label + predicted_label MÂU THUẪN (vd vui bị đoán buon, bất kỳ gì
# lẫn trung_tinh) -> khả năng cao là NHÃN SAI THẬT -> ưu tiên duyệt trước.

INCOMPATIBLE_PAIRS = {frozenset(p) for p in [
    ('vui', 'buon'), ('vui', 'lo_au'), ('vui', 'gian_du'), ('vui', 'trung_tinh'), ('vui', 'that_vong'),
    ('buon', 'trung_tinh'), ('buon', 'hy_vong'),
    ('lo_au', 'trung_tinh'),
    ('gian_du', 'trung_tinh'), ('gian_du', 'hy_vong'),
    ('hy_vong', 'trung_tinh'),
    ('that_vong', 'trung_tinh'),
    ('hy_vong', 'that_vong'),
]}

def pair_status(original, predicted):
    if original == predicted:
        return 'trung_du_doan'  # model đoán đúng luôn, ít khi cần đổi
    return 'MAU_THUAN' if frozenset((original, predicted)) in INCOMPATIBLE_PAIRS else 'chong_lan_hop_le'

for path in ['Data/pipeline_artifacts/review_candidates.csv', 'Data/pipeline_artifacts/all_suspects_full.csv']:
    d = pd.read_csv(path)
    d['pair_status'] = d.apply(lambda r: pair_status(r['original_label'], r['predicted_label']), axis=1)
    # MAU_THUAN lên đầu, trong mỗi nhóm vẫn giữ thứ tự nghi ngờ theo label_quality_score
    order = {'MAU_THUAN': 0, 'chong_lan_hop_le': 1, 'trung_du_doan': 2}
    d['_ord'] = d['pair_status'].map(order)
    d = d.sort_values(['_ord', 'label_quality_score']).drop(columns=['_ord'])
    d.to_csv(path, index=False)
    print(f'{path}: {d["pair_status"].value_counts().to_dict()}')

print('\nĐã thêm cột pair_status, MÂU_THUẪN xếp lên đầu review_candidates.csv — duyệt nhóm này trước.')


In [ ]:
# 8c. Áp kết quả duyệt tay (theo id) -> emotion_training.csv
#
# QUY TRÌNH:
#   1) Chạy 8a (K-fold OOF) -> 8b (xuất review_candidates.csv, action đã pre-fill
#      "remove" cho icon_only, còn lại trống).
#   2) Mở review_candidates.csv, điền cột action từng dòng:
#         keep / để trống  -> giữ nguyên nhãn
#         change           -> đổi nhãn, điền nhãn mới vào cột new_label
#         remove           -> xóa dòng khỏi dataset
#   3) Lưu file, chạy cell này -> ghi đè emotion_training.csv (data train chính)
#   4) Chạy lại notebook TỪ ĐẦU để train model thật trên data đã cập nhật.

import os

REVIEWED_PATH = 'Data/pipeline_artifacts/review_candidates.csv'
CLEANED_PATH  = 'Data/active_inputs/emotion_training.csv'

if not os.path.exists(REVIEWED_PATH):
    print('Chưa có review_candidates.csv — chạy cell 8a rồi 8b trước.')
else:
    reviewed = pd.read_csv(REVIEWED_PATH)
    # id = chỉ số dòng của CSV gốc (tạo ở cell 8a) -> match tuyệt đối, không dựa vào text
    base = pd.read_csv(DATA_PATH).reset_index().rename(columns={'index': 'id'})

    to_delete, relabels, n_invalid = set(), {}, 0
    for _, r in reviewed.iterrows():
        action = str(r.get('action', '')).strip().lower()
        if action in ('', 'nan', 'keep'):
            continue
        rid = int(r['id'])
        if action == 'remove':
            to_delete.add(rid)
        elif action == 'change':
            new_label = str(r.get('new_label', '')).strip().lower()
            if new_label in label2id:
                relabels[rid] = new_label
            else:
                n_invalid += 1
                print(f'  ⚠️ new_label không hợp lệ "{new_label}" (id={rid}): {str(r["text_raw"])[:60]}')
        else:
            n_invalid += 1
            print(f'  ⚠️ action không hợp lệ "{action}" (id={rid}) — chỉ nhận keep/change/remove')

    cleaned = base[~base['id'].isin(to_delete)].copy()
    for rid, new_label in relabels.items():
        cleaned.loc[cleaned['id'] == rid, 'emotion'] = new_label
    cleaned = cleaned.drop(columns=['id'])

    cleaned.to_csv(CLEANED_PATH, index=False)
    print(f'\nĐã xóa {len(to_delete)} dòng, đổi nhãn {len(relabels)} dòng'
          + (f', {n_invalid} dòng lỗi bị bỏ qua' if n_invalid else '') + '.')
    print(f'Xuất: {CLEANED_PATH} ({len(cleaned)}/{len(base)} dòng)')
    print(f'Phân bố nhãn sau làm sạch:')
    print(cleaned['emotion'].value_counts().to_string())
    print(f'\n-> Chạy lại notebook từ đầu để train trên data vừa cập nhật.')


In [31]:
# 9. Inference (Soft Multi-label)
import torch.nn.functional as F

INCOMPATIBLE = [
    ('vui', 'buon'), ('vui', 'lo_au'), ('vui', 'that_vong'),
    ('vui', 'gian_du'), ('hy_vong', 'buon'), ('hy_vong', 'gian_du'),
    ('hy_vong', 'that_vong'),
]

def predict_emotion(sentence, abs_threshold=0.15, gap_threshold=0.40):
    # ✅ FIX: dùng NGUYÊN VĂN, khớp đúng cách xử lý lúc train (cell 3) —
    # không strip emoji nữa (trước đây gọi clean_text(..., label=None) sẽ xóa
    # CẢ 2 chiều emoji, khác hẳn cách train chỉ xóa 1 chiều theo nhãn -> lệch
    # phân phối train/inference).
    raw_sentence = str(sentence)
    sentence = raw_sentence.strip()
    if 'phobert' in MODEL_NAME.lower():
        sentence = ViTokenizer.tokenize(sentence)
    inputs = tokenizer(sentence, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
        
        # Rule-based override: Buff điểm cho từ khóa xuất hiện tường minh
        raw_lower = raw_sentence.lower()
        if 'thất vọng' in raw_lower or 'thất zọng' in raw_lower:
            logits[0, label2id['that_vong']] += 1.5
        if any(w in raw_lower for w in ['lo lắng', 'lo âu', 'căng thẳng', 'stress']):
            logits[0, label2id['lo_au']] += 1.0
            
        probs = F.softmax(logits, dim=1).squeeze().tolist()

    sorted_probs = sorted(enumerate(probs), key=lambda x: -x[1])
    top1_id, p1 = sorted_probs[0]
    top2_id, p2 = sorted_probs[1]

    candidates = [id2label[top1_id]]
    if p2 >= abs_threshold and (p1 - p2) <= gap_threshold:
        candidates.append(id2label[top2_id])

    # Lọc cặp mâu thuẫn
    for a, b in INCOMPATIBLE:
        if a in candidates and b in candidates:
            if probs[label2id[a]] > probs[label2id[b]]:
                candidates.remove(b)
            else:
                candidates.remove(a)
    return candidates

tests = [
    'Dạo này áp lực công việc làm tôi căng thẳng quá, đêm nào cũng mất ngủ',
    'Tuyệt vời! Chúc mừng đội tuyển Việt Nam! :)))',
    'Tôi cảm thấy rất trống rỗng và không muốn làm gì cả.',
    'Thất vọng quá, mong đợi nhiều mà được cái này huhu',
]
for s in tests:
    print(f'{str(predict_emotion(s)):25s} | {s}')


['lo_au']                 | Dạo này áp lực công việc làm tôi căng thẳng quá, đêm nào cũng mất ngủ
['vui']                   | Tuyệt vời! Chúc mừng đội tuyển Việt Nam! :)))
['buon', 'that_vong']     | Tôi cảm thấy rất trống rỗng và không muốn làm gì cả.
['that_vong']             | Thất vọng quá, mong đợi nhiều mà được cái này huhu
